<a href="https://colab.research.google.com/github/Anjali2000702/CP-Lab/blob/main/CP_end__sem_practical1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Mon Jun  1 05:32:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install numba numpy

In [3]:
import numpy as np
from numba import cuda
import math
import time

In [4]:
N = 10_000_000

input_array = np.random.rand(N).astype(np.float32)

print("Dataset Size:", len(input_array))
print(input_array[:5])

Dataset Size: 10000000
[0.299262   0.6015097  0.1578933  0.3941854  0.08051689]


In [5]:
print(cuda.is_available())

True


In [6]:
@cuda.jit
def reverse_array(input_array, output_array, n):
    idx = cuda.grid(1)

    if idx < n:
        output_array[idx] = input_array[n - 1 - idx]

In [7]:
# Copy input array to GPU
d_input = cuda.to_device(input_array)

# Create output array on GPU
d_output = cuda.device_array_like(input_array)

print("GPU memory allocated successfully")

GPU memory allocated successfully


In [8]:
n = len(input_array)

threads_per_block = 256
blocks_per_grid = (n + threads_per_block - 1) // threads_per_block

reverse_array[blocks_per_grid, threads_per_block](
    d_input,
    d_output,
    n
)

cuda.synchronize()

print("Kernel execution completed")

Kernel execution completed


In [9]:
output_array = d_output.copy_to_host()

print("First 5 elements of original array:")
print(input_array[:5])

print("\nLast 5 elements of original array:")
print(input_array[-5:])

print("\nFirst 5 elements of reversed array:")
print(output_array[:5])

print("\nLast 5 elements of reversed array:")
print(output_array[-5:])

First 5 elements of original array:
[0.299262   0.6015097  0.1578933  0.3941854  0.08051689]

Last 5 elements of original array:
[0.22489615 0.451228   0.74233973 0.5342056  0.739567  ]

First 5 elements of reversed array:
[0.739567   0.5342056  0.74233973 0.451228   0.22489615]

Last 5 elements of reversed array:
[0.08051689 0.3941854  0.1578933  0.6015097  0.299262  ]


In [10]:
import time

start = time.time()

reverse_array[blocks_per_grid, threads_per_block](
    d_input,
    d_output,
    n
)

cuda.synchronize()

end = time.time()

print("GPU Execution Time:", end - start, "seconds")

GPU Execution Time: 0.0008003711700439453 seconds


In [11]:
import time

start = time.time()

cpu_output = input_array[::-1]

end = time.time()

print("CPU Execution Time:", end - start, "seconds")


CPU Execution Time: 6.365776062011719e-05 seconds


Array reversal is a simple memory-bound operation. NumPy's slicing is already highly optimized. For such simple operations, CUDA kernel launch overhead can make the GPU slower. GPUs show greater advantages for computationally intensive tasks such as matrix multiplication, deep learning, and large-scale reductions."